# Scale Dictionary Block-wise to Group-wise Conversion

This notebook converts block-wise scale parameters to group-wise by repeating each row 16 times.


In [1]:
import torch
import numpy as np
from pathlib import Path
import shutil

In [3]:
# 设置文件路径
scale_dict_path = "/mnt/nfs/home/zihanm/deepcompressor/sdxl_nvfp4_block/scale.pt"
output_path = "/mnt/nfs/home/zihanm/deepcompressor/sdxl_nvfp4_block/scale_blockwise.pt"

# 加载原始权重文件
print(f"加载权重文件: {scale_dict_path}")
scale_dict = torch.load(scale_dict_path, map_location="cpu")

print(f"权重文件类型: {type(scale_dict)}")
if isinstance(scale_dict, dict):
    print(f"权重文件键的数量: {len(scale_dict)}")
    print("键列表:")
    for i, key in enumerate(scale_dict.keys()):
        print(f"  {i+1}. {key}")
else:
    print(f"权重文件不是字典，而是: {type(scale_dict)}")


加载权重文件: /mnt/nfs/home/zihanm/deepcompressor/sdxl_nvfp4_block/scale.pt
权重文件类型: <class 'dict'>
权重文件键的数量: 2166
键列表:
  1. down_blocks.1.attentions.0.proj_in.weight.scale.0
  2. down_blocks.1.attentions.0.proj_in.weight.scale.1
  3. down_blocks.1.attentions.0.proj_in.weight.zero
  4. down_blocks.1.attentions.0.transformer_blocks.0.attn1.to_q.weight.scale.0
  5. down_blocks.1.attentions.0.transformer_blocks.0.attn1.to_q.weight.scale.1
  6. down_blocks.1.attentions.0.transformer_blocks.0.attn1.to_q.weight.zero
  7. down_blocks.1.attentions.0.transformer_blocks.0.attn1.to_k.weight.scale.0
  8. down_blocks.1.attentions.0.transformer_blocks.0.attn1.to_k.weight.scale.1
  9. down_blocks.1.attentions.0.transformer_blocks.0.attn1.to_k.weight.zero
  10. down_blocks.1.attentions.0.transformer_blocks.0.attn1.to_v.weight.scale.0
  11. down_blocks.1.attentions.0.transformer_blocks.0.attn1.to_v.weight.scale.1
  12. down_blocks.1.attentions.0.transformer_blocks.0.attn1.to_v.weight.zero
  13. down_blocks.1.

In [4]:
# 查看原始权重的形状信息
if isinstance(scale_dict, dict):
    print("=== 原始权重形状信息 ===")
    for key, value in scale_dict.items():
        if isinstance(value, torch.Tensor):
            print(f"{key}:")
            print(f"  形状: {value.shape}")
            print(f"  数据类型: {value.dtype}")
            print(f"  设备: {value.device}")
            print(f"  第一维大小: {value.shape[0] if len(value.shape) > 0 else 'N/A'}")
            print()
else:
    print("权重文件不是字典格式")


=== 原始权重形状信息 ===
down_blocks.1.attentions.0.proj_in.weight.scale.0:
  形状: torch.Size([1, 1, 1, 1])
  数据类型: torch.float32
  设备: cpu
  第一维大小: 1

down_blocks.1.attentions.0.proj_in.weight.scale.1:
  形状: torch.Size([40, 1, 40, 1])
  数据类型: torch.float32
  设备: cpu
  第一维大小: 40

down_blocks.1.attentions.0.proj_in.weight.zero:
  形状: torch.Size([])
  数据类型: torch.float32
  设备: cpu
  第一维大小: N/A

down_blocks.1.attentions.0.transformer_blocks.0.attn1.to_q.weight.scale.0:
  形状: torch.Size([1, 1, 1, 1])
  数据类型: torch.float32
  设备: cpu
  第一维大小: 1

down_blocks.1.attentions.0.transformer_blocks.0.attn1.to_q.weight.scale.1:
  形状: torch.Size([40, 1, 40, 1])
  数据类型: torch.float32
  设备: cpu
  第一维大小: 40

down_blocks.1.attentions.0.transformer_blocks.0.attn1.to_q.weight.zero:
  形状: torch.Size([])
  数据类型: torch.float32
  设备: cpu
  第一维大小: N/A

down_blocks.1.attentions.0.transformer_blocks.0.attn1.to_k.weight.scale.0:
  形状: torch.Size([1, 1, 1, 1])
  数据类型: torch.float32
  设备: cpu
  第一维大小: 1

down_blocks.1.attenti

In [5]:
def convert_blockwise_to_groupwise(scale_dict, default_repeat_factor=16, norm_repeat_factor=64):
    """
    将block-wise的scale参数转换为group-wise参数
    通过将每一行重复repeat_factor次来实现
    只有第一维大小存在且不是1时才进行修改
    
    特殊规则：如果键名中包含"norm"，使用norm_repeat_factor（默认32），否则使用default_repeat_factor（默认16）
    
    Args:
        scale_dict: 原始权重字典
        default_repeat_factor: 默认重复因子，默认为16
        norm_repeat_factor: norm层的重复因子，默认为32
    
    Returns:
        转换后的权重字典
    """
    converted_dict = {}
    
    for key, value in scale_dict.items():
        # 判断是否是norm层，选择合适的重复因子
        if 'norm' in key.lower():
            repeat_factor = norm_repeat_factor
        else:
            repeat_factor = default_repeat_factor
        
        if isinstance(value, torch.Tensor):
            print(f"转换 {key}: {value.shape} -> ", end="")
            
            # 检查张量维度
            if len(value.shape) == 0:
                # 标量张量，直接复制
                converted_dict[key] = value.clone()
                print(f"{value.shape} (标量，保持不变)")
            elif len(value.shape) >= 1 and value.shape[0] == 1:
                # 第一维大小为1，直接复制
                converted_dict[key] = value.clone()
                print(f"{value.shape} (第一维为1，保持不变)")
            elif len(value.shape) >= 1 and value.shape[0] > 1:
                # 第一维大小存在且大于1，进行重复操作
                print(f"(factor={repeat_factor}) ", end="")
                
                if len(value.shape) == 1:
                    # 1D张量，将每个元素重复repeat_factor次
                    repeated_values = []
                    for val in value:
                        repeated_values.extend([val] * repeat_factor)
                    converted_dict[key] = torch.tensor(repeated_values, dtype=value.dtype)
                    print(f"{converted_dict[key].shape}")
                else:
                    # 多维张量，将第一维的每一行重复repeat_factor次
                    # 使用repeat_interleave来重复每一行
                    converted_dict[key] = value.repeat_interleave(repeat_factor, dim=0)
                    print(f"{converted_dict[key].shape}")
            else:
                # 其他情况，直接复制
                converted_dict[key] = value.clone()
                print(f"{value.shape} (其他情况，保持不变)")
        else:
            # 非张量值，直接复制
            converted_dict[key] = value
            print(f"{key}: 非张量值，保持不变")
    
    return converted_dict

# 执行转换
print("=== 开始转换 ===")
print("规则: norm层使用factor=64, 其他层使用factor=16")
print()
converted_scale_dict = convert_blockwise_to_groupwise(scale_dict, norm_repeat_factor=64)
print("\n转换完成！")


=== 开始转换 ===
规则: norm层使用factor=64, 其他层使用factor=16

转换 down_blocks.1.attentions.0.proj_in.weight.scale.0: torch.Size([1, 1, 1, 1]) -> torch.Size([1, 1, 1, 1]) (第一维为1，保持不变)
转换 down_blocks.1.attentions.0.proj_in.weight.scale.1: torch.Size([40, 1, 40, 1]) -> (factor=16) torch.Size([640, 1, 40, 1])
转换 down_blocks.1.attentions.0.proj_in.weight.zero: torch.Size([]) -> torch.Size([]) (标量，保持不变)
转换 down_blocks.1.attentions.0.transformer_blocks.0.attn1.to_q.weight.scale.0: torch.Size([1, 1, 1, 1]) -> torch.Size([1, 1, 1, 1]) (第一维为1，保持不变)
转换 down_blocks.1.attentions.0.transformer_blocks.0.attn1.to_q.weight.scale.1: torch.Size([40, 1, 40, 1]) -> (factor=16) torch.Size([640, 1, 40, 1])
转换 down_blocks.1.attentions.0.transformer_blocks.0.attn1.to_q.weight.zero: torch.Size([]) -> torch.Size([]) (标量，保持不变)
转换 down_blocks.1.attentions.0.transformer_blocks.0.attn1.to_k.weight.scale.0: torch.Size([1, 1, 1, 1]) -> torch.Size([1, 1, 1, 1]) (第一维为1，保持不变)
转换 down_blocks.1.attentions.0.transformer_blocks.0.attn1.

In [6]:
# 验证转换结果
print("=== 转换后权重形状信息 ===")
for key, value in converted_scale_dict.items():
    if isinstance(value, torch.Tensor):
        print(f"{key}:")
        print(f"  形状: {value.shape}")
        print(f"  数据类型: {value.dtype}")
        print(f"  设备: {value.device}")
        print()

# 比较转换前后的差异
print("=== 转换前后对比 ===")
for key in scale_dict.keys():
    if isinstance(scale_dict[key], torch.Tensor) and isinstance(converted_scale_dict[key], torch.Tensor):
        original_shape = scale_dict[key].shape
        converted_shape = converted_scale_dict[key].shape
        
        print(f"{key}:")
        print(f"  原始形状: {original_shape}")
        print(f"  转换后形状: {converted_shape}")
        
        if len(original_shape) > 0 and len(converted_shape) > 0:
            expansion_factor = converted_shape[0] / original_shape[0]
            print(f"  第一维扩展倍数: {expansion_factor}")
        print()


=== 转换后权重形状信息 ===
down_blocks.1.attentions.0.proj_in.weight.scale.0:
  形状: torch.Size([1, 1, 1, 1])
  数据类型: torch.float32
  设备: cpu

down_blocks.1.attentions.0.proj_in.weight.scale.1:
  形状: torch.Size([640, 1, 40, 1])
  数据类型: torch.float32
  设备: cpu

down_blocks.1.attentions.0.proj_in.weight.zero:
  形状: torch.Size([])
  数据类型: torch.float32
  设备: cpu

down_blocks.1.attentions.0.transformer_blocks.0.attn1.to_q.weight.scale.0:
  形状: torch.Size([1, 1, 1, 1])
  数据类型: torch.float32
  设备: cpu

down_blocks.1.attentions.0.transformer_blocks.0.attn1.to_q.weight.scale.1:
  形状: torch.Size([640, 1, 40, 1])
  数据类型: torch.float32
  设备: cpu

down_blocks.1.attentions.0.transformer_blocks.0.attn1.to_q.weight.zero:
  形状: torch.Size([])
  数据类型: torch.float32
  设备: cpu

down_blocks.1.attentions.0.transformer_blocks.0.attn1.to_k.weight.scale.0:
  形状: torch.Size([1, 1, 1, 1])
  数据类型: torch.float32
  设备: cpu

down_blocks.1.attentions.0.transformer_blocks.0.attn1.to_k.weight.scale.1:
  形状: torch.Size([640, 1, 

In [7]:
# 保存转换后的权重文件
print(f"保存转换后的权重文件到: {output_path}")
torch.save(converted_scale_dict, output_path)

# 验证保存的文件
print("\n验证保存的文件...")
loaded_converted = torch.load(output_path, map_location="cpu")
print(f"保存的文件类型: {type(loaded_converted)}")
print(f"保存的文件键数量: {len(loaded_converted) if isinstance(loaded_converted, dict) else 'N/A'}")

# 检查保存的文件是否正确
if isinstance(loaded_converted, dict):
    print("\n保存文件的形状:")
    for key, value in loaded_converted.items():
        if isinstance(value, torch.Tensor):
            print(f"  {key}: {value.shape}")

print("\n转换和保存完成！")


保存转换后的权重文件到: /mnt/nfs/home/zihanm/deepcompressor/sdxl_nvfp4_block/scale_blockwise.pt

验证保存的文件...
保存的文件类型: <class 'dict'>
保存的文件键数量: 2166

保存文件的形状:
  down_blocks.1.attentions.0.proj_in.weight.scale.0: torch.Size([1, 1, 1, 1])
  down_blocks.1.attentions.0.proj_in.weight.scale.1: torch.Size([640, 1, 40, 1])
  down_blocks.1.attentions.0.proj_in.weight.zero: torch.Size([])
  down_blocks.1.attentions.0.transformer_blocks.0.attn1.to_q.weight.scale.0: torch.Size([1, 1, 1, 1])
  down_blocks.1.attentions.0.transformer_blocks.0.attn1.to_q.weight.scale.1: torch.Size([640, 1, 40, 1])
  down_blocks.1.attentions.0.transformer_blocks.0.attn1.to_q.weight.zero: torch.Size([])
  down_blocks.1.attentions.0.transformer_blocks.0.attn1.to_k.weight.scale.0: torch.Size([1, 1, 1, 1])
  down_blocks.1.attentions.0.transformer_blocks.0.attn1.to_k.weight.scale.1: torch.Size([640, 1, 40, 1])
  down_blocks.1.attentions.0.transformer_blocks.0.attn1.to_k.weight.zero: torch.Size([])
  down_blocks.1.attentions.0.transform